# Week 3: QR faktorizace matice

**Assignment:** [![Run in JupyterHub](https://img.shields.io/badge/Run%20in-JupyterHub-F37626?logo=jupyter)](https://hpc.troja.mff.cuni.cz:8000/hub/user-redirect/git-pull?app=lab&repo=https%3A%2F%2Fgitlab.karlin.mff.cuni.cz%2Fznm%2Fcviceni-student&branch=main&subpath=notebooks%2Fassignment%2Fweek-03-qr-decomposition.ipynb) [![Download .ipynb](https://img.shields.io/badge/Download-.ipynb-blue)](https://gitlab.karlin.mff.cuni.cz/znm/cviceni-student/-/raw/main/notebooks/assignment/week-03-qr-decomposition.ipynb?inline=false) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/num-math/cviceni-student/blob/main/notebooks/assignment/week-03-qr-decomposition.ipynb)

## Importy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(123456789123456789)


def test_qr_decomposition(qr_func, func_name, shapes=None, tol=1e-8):
    """
    Test funkce pro QR rozklad. Pouziva se pro vsechny 4 algoritmy
    (CGS, MGS, Householder, Givens), protoze vsechny musi splnovat stejne vlastnosti.

    Kontroluje se (v tomto poradi):
      1) vstupni matice A neni uvnitr funkce prepsana
      2) tvary matic Q a R odpovidaji ocekavanym rozmerum
      3) Q a R neobsahuji NaN/Inf
      4) R je horni trojuhelnikova (prvky pod diagonalou jsou ~0)
      5) Q je ortogonalni, tj. Q^T Q ~ I
      6) plati A = Q@R (relativni reziduum)
      7) Q,R se (az na znamenko sloupcu/radku) shoduji s referencnim vysledkem
         "numpy.linalg.qr(A)" - QR rozklad je pro matici s plnou sloupcovou hodnosti
         jednoznacny az na soucasne prehozeni signs sloupce Q_i a radku R_i.

    Testuje se na nekolika ruznych tvarech A (mala/velka, ctvercova, na vysku i na sirku),
    Chyba se casto projevi jen pro nektery specialni tvar (napr. m < n).
    """
    if shapes is None:
        shapes = np.array([(3, 3), (5, 5), (8, 5), (5, 8), (50, 20)])

    rng = np.random.default_rng(42)

    for idx in range(0, len(shapes)):
        m, n = shapes[idx]

        A = rng.random((m, n))
        A_original = np.copy(A)

        Q, R = qr_func(A)

        # 1) A nesmi byt zmenena
        assert np.allclose(A, A_original), (
            f"[{func_name}] test na matici {m}-krat-{n}: puvodni matice A byla uvnitr funkce prepsana. Typicka pricina: pouzivate primo 'z = A[:,j]' (coz je jen odkaz do pameti A, nikoliv kopie) misto 'z = np.copy(A[:,j])', a pak menite 'z' na miste (napr. 'z -= ...'), cimz nechtene menite i A - viz varovani '\"Pozor na python!\"' v Uloze A1."
        )

        k = min(m, n)

        # 2) rozmery Q a R
        assert Q.shape == (m, k), (
            f"[{func_name}] test na matici {m}-krat-{n}: Q ma tvar {Q.shape}, ale ocekavany tvar je ({m}, {k}). Zkontrolujte alokaci 'Q = np.zeros((m, min(m,n)))' a ze zapisujete do vsech jejich sloupcu."
        )
        assert R.shape == (k, n), (
            f"[{func_name}] test na matici {m}-krat-{n}: R ma tvar {R.shape}, ale ocekavany tvar je ({k}, {n}). Zkontrolujte alokaci 'R = np.zeros((min(m,n), n))'."
        )

        # 3) NaN / Inf
        assert np.all(np.isfinite(Q)) and np.all(np.isfinite(R)), (
            f"[{func_name}] test na matici {m}-krat-{n}: Q nebo R obsahuje NaN nebo Inf. Nejcastejsi pricina je deleni nulovou normou (napr. 'Q[:,j] = z / R[j,j]' kdyz vyjde 'R[j,j] == 0'), typicky kdyz je sloupec A (skoro) linearne zavisly na predchozich sloupcich."
        )

        # 4) R horni trojuhelnikova
        below_diagonal = np.tril(R, -1)
        triangularity_error = np.max(np.abs(below_diagonal)) if below_diagonal.size > 0 else 0.0
        assert triangularity_error < tol, (
            f"[{func_name}] test na matici {m}-krat-{n}: R neni horni trojuhelnikova, nejvetsi prvek pod diagonalou ma velikost {triangularity_error:.2e} (cekali jsme < {tol:.0e}). Typicka pricina: spatne meze/indexy v cyklu pri projekci nebo eliminaci (napr. zamenene 'i' a 'j'), nebo se u Householdera/Givense transformace neaplikuje na spravnou podmatici (napr. 'R[k:, :]' misto cele R, resp. spravny par radku 'R[i-1:i+1, :]')."
        )

        # 5) Q ortogonalni
        orthogonality_loss = np.linalg.norm(np.eye(k) - Q.T @ Q, ord='fro')
        assert orthogonality_loss < 1e-6, (
            f"[{func_name}] test na matici {m}-krat-{n}: sloupce Q nejsou ortonormalni, ||I - Q^T Q||_F = {orthogonality_loss:.2e}  (cekali jsme < 1e-6. Typicka pricina: chybejici nebo spatne provedena normalizace ('Q[:,j] = z / R[j,j]'), nebo se pri ortogonalizaci neodecitaji vsechny projekce na uz spoctene sloupce (Gram-Schmidt - klasicka vs. modifikovana verze pletou 'z' a puvodni sloupec A), pripadne je spatne sestavena reflexe/rotace (Householder/Givens)."
        )

        # 6) rekonstrukce A = QR
        A_norm = np.linalg.norm(A_original, ord='fro')
        relative_residual = np.linalg.norm(A_original - Q @ R, ord='fro') / A_norm
        assert relative_residual < 1e-6, (
            f"[{func_name}] test na matici {m}-krat-{n}: A != Q@R, relativni reziduum ||A - QR||_F / ||A||_F = {relative_residual:.2e} (cekali jsme < 1e-6). I kdyz jsou Q ortogonalni a R horni trojuhelnikova, soucin nedava puvodni matici - zkontrolujte poradi nasobeni transformacnich matic (Q se musi skladat ve spravnem poradi), nebo znamenkovou konvenci pri konstrukci Householderova vektoru nebo Givensovy rotace."
        )

        # 7) shoda s referencnim QR rozkladem numpy (az na znamenko)
        Q_ref, R_ref = np.linalg.qr(A_original, mode='reduced')
        signs = np.sign(np.sum(Q * Q_ref, axis=0))
        signs[signs == 0] = 1.0
        Q_error_vs_ref = np.max(np.abs(Q * signs[np.newaxis, :] - Q_ref))
        R_error_vs_ref = np.max(np.abs(R * signs[:, np.newaxis] - R_ref))
        assert max(Q_error_vs_ref, R_error_vs_ref) < 1e-6, (
            f"[{func_name}] test na matici {m}-krat-{n}: Q/R se (az na znamenko sloupcu/radku) neshoduji s referencnim vysledkem 'numpy.linalg.qr(A)' (rozdil = {max(Q_error_vs_ref, R_error_vs_ref):.2e}). I kdyz vase Q,R mozna vypadaji rozumne (jsou priblizne ortogonalni a trojuhelnikove), nejde o spravny QR rozklad teto konkretni matice A - zkontrolujte cely algoritmus (napr. jestli jste nahodou nezamichali jiny rozklad, jako LU, nebo jestli neskladate transformace/sloupce ve spatnem poradi)."
        )

    return f"test_{func_name} OK (otestovano na tvarech {shapes})"

## Materiály

- [poznámky z přednášky](https://www.karlin.mff.cuni.cz/~blechta/znm/assets/OG_transformace.pdf)
- další materiály
  - kniha [Duintjer Tebbens a kolektiv: Analýza metod pro maticové výpočty](https://cuni.primo.exlibrisgroup.com/discovery/fulldisplay?docid=alma990020832750106986&context=L&vid=420CKIS_INST:UKAZ&lang=cs&search_scope=MyInst_and_CI&adaptor=Local%20Search%20Engine&tab=Everything&query=any,contains,Anal%C3%BDza%20metod%20pro%20maticov%C3%A9%20v%C3%BDpo%C4%8Dty&offset=0) (Kapitola 3)
  - skripta [Barto, Tůma: Lineární algebra](https://www.mff.cuni.cz/data/web/obsah/department_math/ka/skripta_la7.pdf) (Sekce 8.4)
  - skripta [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (Volume 1, Kapitola 4)

- ***Primární úlohy*** : A1, B1, B2

- ***Úlohy k bonusové části zkoušky*** : B1, B2, C1, C2

- základní ***Learning outcomes*** (cviko, nikoliv přednáška)
  - Umím vysvětlit rozdíl mezi klasickou a modifikovanou implementací Gram-Schmidtova procesu.
  - Umím porovnat výpočetní náročnost i kvalitu 4 postupů pro $QR$ faktorizaci matice (cGS, mGS, Householder, Givens).
  - Umím vysvětlit 2 ekvivalentní postupy použití $QR$ faktorizace matice k vyřešení $A\mathbf{x}=\mathbf{b}$ a porovnat jejich efektivitu.
  - Umím porovnat použití $QR$ faktorizace a $LU$ faktorizace coby nástrojů k řešení $A\mathbf{x}=\mathbf{b}$.

## Intro

V [cvičení o iteračních metodách](week-02-iterative-methods) jsme viděli, že Gaussova eliminace s částečnou pivotací (a jí ekvivalentní algoritmus LU-faktorizace s částečnou pivotací) jsou numericky pouze ***podmíněně stabilní*** - tedy může se stát, že výsledky jimi spočtené jsou naprosto špatně. Přestože se v mnoha aplikacích tyto metody používají, je vhodné znát i metody, které jsou ***nepodmíněně stabilní***.

Viděli jsme, že nestabilita Gaussovy eliminace pro $A\mathbf{x} = \mathbf{b}$ pramení z používání transformací na $A$, které mohou výrazně měnit normu objektu, který transformují -- v případě Wilkinsonovy matice tyto transformace odpovídali sčítání prvků ve vektoru.

Proto dává smysl snažit se namísto těchto používat transformace ***unitární***, tj. takové, které normu nemění vůbec. Odtud tedy plyne nápad, že by bylo dobré umět spočítat [***QR-faktorizaci***](https://en.wikipedia.org/wiki/QR_decomposition) matice $A$, tj, spočíst matice $Q$ a $R$ takové, že

\begin{equation*} A = QR \end{equation*}

a zároveň

  - $Q$ je ***unitární***, tj. $\forall \, \mathbf{v} \,:\, \|Q\mathbf{v}\|_2 = \|\mathbf{v}\|_2 $,
  - $R$ je ***horní trojúhelníková***.


Na přednášce jsme viděli celkem 4 algoritmy, které tuto faktorizaci počítají.

## Část A - [Gram-Schmidt](https://en.wikipedia.org/wiki/Gram%E2%80%93Schmidt_process#)

>Je to algoritmus, který znáte z Lingebry 1, kde jste ho používali na ortogonalizaci posloupnosti vektorů. Jako aplikaci jste si zmiňovali právě i QR-faktorizaci matice a souvislost s výpočtem ortonormální báze prostoru $\mathrm{Im}(A)$. Tento algoritmus má několik verzí a my jsme se bavili o [klasické a modifikované verzi](https://dspace.mit.edu/bitstream/handle/1721.1/75282/18-335j-fall-2006/contents/lecture-notes/lec5handout6pp.pdf). Princip obou verzí je stejný - vezmeme si sloupce matice $A$ jako bázi prostoru $\mathrm{Im}(A)$ a postupně vektory této báze ortogonalizujeme vůči předchozím pomocí odčítání projekcí na již spočtenou část báze.
>
>Pokud již máme spočítaných prvních $k$ vektorů $\mathbf{q}_1, \ldots, \mathbf{q}_{k}$ pomocí Gram-Schmidta, tj. platí
>
>\begin{equation*}
\mathbf{q}_1,\dotsc , \mathbf{q}_k \mathrm{\; tvori \; ON \; bazi}
\qquad \& \qquad
\mathrm{span} \{A_{:,1},\dotsc ,A_{:,k}\} = \mathrm{span} \{ \mathbf{q}_1,\dotsc , \mathbf{q}_k \},
\end{equation*}
>
>pak $\mathbf{q}_{k+1}$ získáme jako
>
>\begin{equation*}
\mathbf{z} :=
A_{:,k+1} - \sum\limits_{i = 1}^{k} \mathbf{q}_i \mathbf{q}_i^T A_{:,k+1} =
\underbrace{ \, \left(I-\sum\limits_{i = 1}^{k} \mathbf{q}_i \mathbf{q}_i^T \right) A_{:,k+1} \,}_{\mathrm{klasicky \; GS}} \; = \;
\underbrace{ \, \prod\limits_{i=1}^{k} \left( I - \mathbf{q}_i \mathbf{q}_i^T \right) A_{:,k+1} \,}_{\mathrm{modifikovany \; GS}}
\quad \& \quad
\mathbf{q}_{k+1} = \mathbf{z} / \|\mathbf{z}\|_2.
\end{equation*}
>
>V ***klasické verzi*** celý algoritmus stojí na počítání skalárních součinů
>\begin{equation*}
r_{i,k+1} := \mathbf{q}_i^T A_{:,k+1}
\qquad \mathrm{a \; pak} \qquad
\mathbf{z} := A_{:,k+1} - \sum\limits_{i = 1}^{k} \mathbf{q}_i r_{i,k}.
\end{equation*}
>
> Vidíme, že výpočet všech skalárních součinů, tj. výpočet všech koeficientů $r_{i,k}$, může probíhat nezávisle na sobě, tj. *paralelně*.
>
>V ***modifikované verzi*** vše stojí na *sekvenčním* výpočtu
>
> - $r_{1,k+1} := \mathbf{q}_1^T A_{:,k+1} \quad \mathrm{a} \quad \mathbf{v}_1:=A_{:,k+1} - \mathbf{q}_{1}r_{1,k+1}$,
> - $r_{2,k+1} := \mathbf{q}_2^T \mathbf{v}_1 \quad \mathrm{a} \quad \mathbf{v}_2:= \mathbf{v}_1 - \mathbf{q}_2 r_{2,k+1}$,
> - $r_{3,k+1} := \mathbf{q}_3^T \mathbf{v}_2 \quad \mathrm{a} \quad \mathbf{v}_3:= \mathbf{v}_2 - \mathbf{q}_3 r_{2,k+1}$,
> - $\dotsc$
> - $r_{k,k+1} := \mathbf{q}_{k}^T \mathbf{v}_{k-1} \quad \mathrm{a} \quad \mathbf{v}_k:= \mathbf{v}_{k-1} - \mathbf{q}_k r_{k,k+1}$
> - a až nakonec $\mathbf{z} := \mathbf{v}_k$
>
>Označíme-li $r_{k+1,k+1} := \|\mathbf{z}\|_2$, pak dostaneme pro obě verze (stejně jako na přednášce)
>
>\begin{equation*}
A_{:,k+1} = \sum_{i = 1}^{k} r_{i,k+1} \mathbf{q}_i + r_{k+1,k+1} \ \mathbf{q}_{k+1}, \quad k=1,\ldots ,m,
\end{equation*}
>
>nebo-li $A = QR$.

### Úloha A1 - implementace G-S

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Kódění**

Naprogramujte obě varianty výpočtu QR-faktorizace matice $A$ pomocí Gram-Schmidtova ortogonalizačního procesu.
- Pro reálné vektory `u,v` v numpy má
    - skalární součin syntax `v.T @ u` a nebo `np.dot(u,v)` (`u * v` provede násobení po prvcích, pokud to dimenze dovolí).
    - norma syntax `np.linalg.norm(u)` (případně  `np.linalg.norm(u,2)` pro explicitní specifikaci, že nám jde eukleidovskou normu).

  - "Pozor na python!" - pokud máme matici `A`, zadefinujeme si `z` jako její první sloupec `z = A[:,0]` a provedeme kód `z -= 1`, pak ***tím měníme i původní matici `A` - konkrétně `A[:,0]` na `A[:,0] - 1`***. Stejně se chová i příkaz `z[:] = z - 1`.
  - V pythonu příkaz `z = A[:,0]` nevytvoří kopii `A[:,0]` (tj. nealokuje konkrétní bity k uložení `z` a neokopíruje tam hodnoty vektoru `A[:,0]`). Namísto toho si uloží `z` jako "paměťový odkaz" a při každé operaci s `z` provede kýženou operaci s daty uloženými na "daném paměťovém odkazu".
- Pokud chceme ověřit, že pro výsledné matice $Q,R$ platí $A=QR$, pak nechceme přepisovat matici $A$.
  - Pokud chci tomuto předejít můžu
- *(i)* napsat `z = np.copy( A[:,0] )`. To donutí python opravdu alokovat nové bity pro novou proměnnou `z` a tam zkopírovat hodnoty `A[:,0]`. Tímto se zpřetrhá vazba mezi `z` a `A[:,0]` a už si nadále jdou svými vlastními cestami.
    - nebo *(ii)* místo `z -= 1` nebo `z[:] = z - 1` napsat `z = z - 1`. To  také python donutí alokovat nové bity pro novou proměnnou `z`. Tímto se také zpřetrhají vazby mezi `z` a `A[:,0]` a už si nadále jdou svými vlastními cestami.

</blockquote>
</div>

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Úkol**

Doplňte tělo funkce `qr_classical_gram_schmidt()` (klasická verze Gram-Schmidtova ortogonalizačního procesu) dle anotace.

</blockquote>
</div>

In [ ]:
def qr_classical_gram_schmidt(A):
    """
    QR rozklad implementovany pomoci **klasicke** verze Gram-Schmidtovy ortogonalizace.

    Input
    -------
    A : np.array dimenze "m-krat-n" s prvky np.float64. Matice rozmeru m-krat-n

    Output
    -------
    Q : np.array dimenze "m-krat-min(m,n)" s prvky np.float64. Ortogonalni matice, sloupce tvori ON bazi linearniho obalu sloupcu A
    R : np.array dimenze "min(m,n)-krat-n" s prvky np.float64. Horni trojuhelnikova matice tak, aby A = QR
    """
    m, n = A.shape
    Q = np.zeros((m, min(m, n)))
    R = np.zeros((min(m, n), n))

    for j in range(n):
        ...

    return Q, R

#######################################################
test_qr_decomposition(qr_classical_gram_schmidt, "qr_classical_gram_schmidt")

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Úkol**

Doplňte tělo funkce `qr_modified_gram_schmidt()` (modifikovaná verze Gram-Schmidtova ortogonalizačního procesu) dle anotace v docstringu.

</blockquote>
</div>

In [ ]:
def qr_modified_gram_schmidt(A):
    """
    QR rozklad implementovany pomoci **modifikovane** verze Gram-Schmidtovy ortogonalizace.

    Input
    -------
    A : np.array dimenze "m-krat-n" s prvky np.float64. Matice rozmeru m-krat-n

    Output
    -------
    Q : np.array dimenze "m-krat-min(m,n)" s prvky np.float64. Ortogonalni matice, sloupce tvori ON bazi linearniho obalu sloupcu A
    R : np.array dimenze "min(m,n)-krat-n" s prvky np.float64. Horni trojuhelnikova matice tak, aby A = QR
    """
    m, n = A.shape
    Q = np.zeros((m, min(m, n)))
    R = np.zeros((min(m, n), n))

    for j in range(n):
        ...

    return Q, R

##############################################################
test_qr_decomposition(qr_modified_gram_schmidt, "qr_modified_gram_schmidt")

## ZÚ - [Householderovy reflexe](https://en.wikipedia.org/wiki/Householder_transformation)

> Idea je vcelku jednoduchá - jako u Gaussovy eliminace chceme postupně vynulovat všechny nenulové prvky pod diagonálou, sloupec po sloupci. Pokud to ale budeme dělat pomocí ortogonálních transformací (tj. unitárních matic), bude nutně naše transformační matice unitární a tedy dostaneme QR-faktorizaci původní matice.
>
> Transformací, kterou na vynulovávání použijeme, je ***zrcadlení/reflexe*** vektoru - zjevně také nemění normu a je vždy možné najít nadrovinu tak, aby se daný vektor $\mathbf{x}\in\mathbb{R}^n$ zreflektoval na $\|\mathbf{x}\|_2 \cdot \mathbf{e}_1$. Na přednášce jsme si odvodili, že matice, která této reflexi odpovídá je dána jako
>
>
>\begin{equation*}
H = I_n - 2\cdot \mathbf{q}\mathbf{q}^T,
\qquad \mathrm{kde} \qquad
\mathbf{q} := \frac{ \mathbf{x} + \mathrm{sign}(x_1)\cdot \|\mathbf{x}\|_2 \cdot \mathbf{e}_1 }{ \| \mathbf{x} + \mathrm{sign}(x_1)\cdot \|\mathbf{x}\|_2 \cdot \mathbf{e}_1 \|_2 }.
\end{equation*}

### Úloha 1

<div style="border-left: 4px solid #8250df; background-color: #fbefff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**❗ ZÚ (semestrální projekt)**

Naprogramujte výpočet QR-faktorizace matice $A$ pomocí Householderových reflexí doplněním těla funkce `qr_householder()` dle anotace.

</blockquote>
</div>

In [ ]:
def qr_householder(A):
    """
    QR rozklad implementovany pomoci Householderovych reflexi.

    Input
    -------
    A : np.array dimenze "m-krat-n" s prvky np.float64. Matice rozmeru m-krat-n

    Output
    -------
    Q : np.array dimenze "m-krat-m" s prvky np.float64. Ortogonalni matice, sloupce tvori ON bazi linearniho obalu sloupcu A
    R : np.array dimenze "m-krat-n" s prvky np.float64. Horni trojuhelnikova matice tak, aby A = QR
    """
    return

##############################################################
test_qr_decomposition(qr_householder, "qr_householder")

In [ ]:
############################# ne/mám nakóděného Householdera #############################
MamNakodenehoHouseholdera = False
############################# ne/mám nakóděného Householdera #############################

## [Givensovy rotace (navíc)](https://en.wikipedia.org/wiki/Givens_rotation)

Idea je stejná jako u Householderových reflexích - postupně vynulovávat všechny nenulové prvky pod diagonálou, sloupec po sloupci pomocí ortogonálních transformací (tj. unitárních matic).

Transformací, kterou na vynulování použijeme, je ***rotace*** vektoru - rotace zjevně nemění normu a je zjevně vždy možné zrotovat daný vektor $[x_1,x_2]^T$ na vektor, který má jednu nulovou souřadnici (a stejnou normu), tj. na vektor  $[\sqrt{x_1^2 + x_2^2},0]^T$. Vhodnou posloupností rotací (což bude sama o sobě taky rotace) jsme tedy schopni transformovat vektor

\begin{equation*}
\mathbf{x} := \begin{bmatrix} x_1 \\ x_2 \\ x_3 \\ \vdots \\ x_n \end{bmatrix}
\underbrace{\quad \rightarrow \quad}_{\mathrm{rotace\;} R_{1}}
\begin{bmatrix} \sqrt{x_1^2 + x_2^2} \\ 0 \\ x_3 \\ \vdots \\ x_n \end{bmatrix}
\underbrace{\quad \rightarrow \quad}_{\mathrm{rotace\;} R_{2}}
\dotsc
\underbrace{\quad \rightarrow \quad}_{\mathrm{rotace\;} R_{n-1}}
\begin{bmatrix} \|\mathbf{x}\|_2 \\ 0 \\ 0 \\ \vdots \\ 0 \end{bmatrix}.
\end{equation*}

Matice odpovídající rotacím $R_1,\dotsc ,R_{n-1}$ se klasicky značí $G_1,\dotsc ,G_{n-1}$ a mají přímočaře odvoditelný tvar, který jsme si na přednášce ukázali:

\begin{equation*}
G_1 =
\begin{bmatrix}
c_1 & -s_1 \\
s_1 & c_1 \\
&& 1 \\
&&& \ddots \\
&&&& 1
\end{bmatrix}, \quad
G_2 =
\begin{bmatrix}
c_1 && -s_1 \\
 & 1 \\
s_1 && c_1 \\
&&& \ddots \\
&&&& 1
\end{bmatrix}, \quad \dotsc \in \mathbb{R}^{n\times n}.
\end{equation*}

In [ ]:
def qr_givens(A):
    """
    QR rozklad implementovany pomoci Givensovych rotaci.

    Input
    -------
    A : np.array dimenze "m-krat-n" s prvky np.float64. Matice rozmeru m-krat-n

    Output
    -------
    Q : np.array dimenze "m-krat-m" s prvky np.float64. Ortogonalni matice, sloupce tvori ON bazi linearniho obalu sloupcu A
    R : np.array dimenze "m-krat-n" s prvky np.float64. Horni trojuhelnikova matice tak, aby A = QR
    """
    def givens_rotation(a, b):
        """ Vypocitame prvky "c" a "s" pro rotaci [a,b] na [+/-norm([a,b]),0]. Nezapomeňte ošetřit případy jako "a=0" a "b=0" """
        if b == 0:
            c, s = 1, 0
        elif a == 0:
            c, s = 0, 1
        elif abs(b) > abs(a):
            t = -a / b
            s = 1 / np.sqrt(1 + t**2)
            c = s * t
        else:
            t = -b / a
            c = 1 / np.sqrt(1 + t**2)
            s = c * t
        return np.array([[c, -s], [s, c]])

    m, n = A.shape
    Q = np.eye(m)
    R = A.copy()

    for j in range(min(m, n)):
        for i in range(m - 1, j, -1):
            G = givens_rotation(R[i - 1, j], R[i, j])
            rows = np.array([i - 1, i])
            R[rows, :] = G @ R[rows, :]
            Q[:, rows] = Q[:, rows] @ G.T

    return Q[:, :min(m, n)], R[:min(m, n), :]

##############################################################
test_qr_decomposition(qr_givens, "qr_givens")

## Část B: Který algoritmus vybrat?

Výše jsme si připomněli/nakódili několik způsobů jak spočíst $QR$-faktorizaci matice $A$ - přirozená otázka je, který bychom si měli v závislosti na matici $A$ vybrat? Porovnávat budeme
   - kvalitu výsledné $QR$-faktorizace
   - výpočetní náročnost daného postupu $QR$-faktorizace

### Úloha B1 - Kvalita

Dejme tomu, že pro danou matici $A\in \mathbb{R}^{m\times n}$ spočítáme její $QR$-faktorizaci jedním z algoritmů výše a dostaneme matice $\hat{Q}$ a $\hat{R}$.


První rozumný, způsob jak měřit kvalitu $QR$ faktorizace, je tzv. její ***relativní reziduum***, tj. norma $ \frac{ \| A - \hat{Q}\hat{R} \| }{ \| A \| }$.
  - pokud bychom měli velké reziduum, zjevně daný algoritmus nedělá to, co bychom chtěli;

Druhý způsob, jak měřit kvalitu $QR$ faktorizace, je tzv. ***ztráta ortogonality výsledné báze***, tj. norma $\| I - \hat{Q}^T\hat{Q} \|_F$.
  - na druhou stranu jsme chtěli algoritmus, který bude numericky stabilní (ne jako $LU$-faktorizace, aka Gaussova eliminace) - chtěli jsme algoritmus, který vytvoří ortonormální bazi.

Zjevně oba přístupy dávají v principu smysl, přestože měří zcela jiné věci - více o jejich porovnání je např. v [Duintjer Tebbens a kolektiv: Analýza metod pro maticové výpočty](https://cuni.primo.exlibrisgroup.com/discovery/fulldisplay?docid=alma990020832750106986&context=L&vid=420CKIS_INST:UKAZ&lang=cs&search_scope=MyInst_and_CI&adaptor=Local%20Search%20Engine&tab=Everything&query=any,contains,Anal%C3%BDza%20metod%20pro%20maticov%C3%A9%20v%C3%BDpo%C4%8Dty&offset=0)  (Sekce 3.5.5)

In [ ]:
def random_matrix(n, cond):
    """ Vrati nesingularni ctvercovou matici A rozmeru "n"-krat-"n" s cislem podminenosti kappa(A) = "cond". """
    rng = np.random.default_rng(123456789123456789)  ### tenhle radek zajistuje, ze ty nahodnz cisla jsou vygenerovanz pri kazdem behu stejne (lze smazat)
    A = rng.standard_normal(size=(n, n))
    U, _, Vt = np.linalg.svd(A)
    S = np.diag(np.logspace(0, -np.log10(cond), n))
    return U @ S @ Vt


cond_numbers = np.logspace(0, 20, 30)
n = 50

residual_cgs = np.zeros(len(cond_numbers))
orth_loss_cgs = np.zeros(len(cond_numbers))
residual_mgs = np.zeros(len(cond_numbers))
orth_loss_mgs = np.zeros(len(cond_numbers))
residual_givens = np.zeros(len(cond_numbers))
orth_loss_givens = np.zeros(len(cond_numbers))
residual_hh = np.zeros(len(cond_numbers))
orth_loss_hh = np.zeros(len(cond_numbers))

I_n = np.eye(n)
for ind in range(len(cond_numbers)):
    A = random_matrix(n, cond_numbers[ind])

    Q, R = qr_classical_gram_schmidt(A)
    residual_cgs[ind] = np.linalg.norm(A - Q @ R, ord='fro')
    orth_loss_cgs[ind] = np.linalg.norm(I_n - Q.T @ Q, ord='fro')

    Q, R = qr_modified_gram_schmidt(A)
    residual_mgs[ind] = np.linalg.norm(A - Q @ R, ord='fro')
    orth_loss_mgs[ind] = np.linalg.norm(I_n - Q.T @ Q, ord='fro')

    Q, R = qr_givens(A)
    residual_givens[ind] = np.linalg.norm(A - Q @ R, ord='fro')
    orth_loss_givens[ind] = np.linalg.norm(I_n - Q.T @ Q, ord='fro')

    if MamNakodenehoHouseholdera:
        Q, R = qr_householder(A)
        residual_hh[ind] = np.linalg.norm(A - Q @ R, ord='fro')
        orth_loss_hh[ind] = np.linalg.norm(I_n - Q.T @ Q, ord='fro')
    else:
        residual_hh[ind] = np.nan; orth_loss_hh[ind] = np.nan

#################################### Vykreslovani
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.loglog(cond_numbers, residual_cgs, 'o:', color='tab:red', markeredgecolor='k', label=r'reziduum CGS')
plt.loglog(cond_numbers, residual_mgs, 'o:', color='tab:blue', markeredgecolor='k', label=r'reziduum MGS')
plt.loglog(cond_numbers, residual_givens, 'o:', color='tab:green', markeredgecolor='k', label=r'reziduum Givens')
if (residual_hh[0:3] != np.zeros(3)).all():
    plt.loglog(cond_numbers, residual_hh, 'o:', color='orange', markeredgecolor='k', label=r'reziduum Householder')
plt.legend(); plt.xlabel(r'$\kappa(A)$')

plt.subplot(1, 2, 2)
plt.loglog(cond_numbers, orth_loss_cgs, 'o:', color='tab:red', markeredgecolor='k', label=r'ztrata OG CGS')
plt.loglog(cond_numbers, orth_loss_mgs, 'o:', color='tab:blue', markeredgecolor='k', label=r'ztrata OG MGS')
plt.loglog(cond_numbers, orth_loss_givens, 'o:', color='tab:green', markeredgecolor='k', label=r'ztrata OG Givens')
if (residual_hh[0:3] != np.zeros(3)).all():
    plt.loglog(cond_numbers, orth_loss_hh, 'o:', color='orange', markeredgecolor='k', label=r'ztrata OG Householder')
plt.legend(); plt.xlabel(r'$\kappa(A)$'); plt.show()
#################################### Vykreslovani

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

Na základě výsledků dovoďte závislost $\| A - \hat{Q}\hat{R}\|_F$ a $\| I - \hat{Q}^T\hat{Q} \|_F$ na $\kappa(A)$ jako explicitní funkci za použití Landau notace ($\mathcal{O}(\dotsc)$). Detailně svou odpověď zdůvodněte/podložte daty z grafu.

</blockquote>
</div>

### ZÚ - Výpočetní náročnost

Dalším způsobem jak porovnat jednotlivé způsoby algoritmy je jejich výpočetní čas.

Pro měření času programu použijeme [`%timeit -o -q`](https://ipython.readthedocs.io/en/stable/interactive/magics.html#magic-timeit) :

  ```
  # nasledujici kod zmeri cas potrebny na exekuci funkce "MojeFunkce()"
  TimeResult = %timeit -o -q MojeFunkce_output = MojeFunkce(MojeFunkce_input)
  print('Prumerny cas na exekuci "MojeFunkce()" je', TimeResult.average)
  print('Smerodatna odchylka exekuce "MojeFunkce()" je', TimeResult.stdev)

  # Funkce "%timeit" mnohokrat spusti zbytek radku po prikazu "%timeit" a udela jednoduchou statistiku casu.
  # Meren je cas celeho zbyleho radku, tj. vseho za prikazem "%timeit".
  # Obecne funkce "%timeit" vraci objekt (tj. nikoliv pouze cas).
  # V tomto objektu je property "average" - tj. "TimeResult.average" udava prumerny cas na exekuci.
  # flag "-o" zajistuje, ze "%timeit" vrati objekt `TimeitResult`
  # flag "-q" zajistuje, ze "%timeit" bude zticha
  ```

In [ ]:
n_values = np.logspace(1, np.log10(500), 6, dtype=np.int32)

times_cgs = np.zeros(len(n_values))
times_mgs = np.zeros(len(n_values))
times_givens = np.zeros(len(n_values))
times_hh = np.zeros(len(n_values))

for ind in range(len(n_values)):
    n = n_values[ind]
    A = random_matrix(n, 10)

    time_result = %timeit -o -q Q,R = qr_classical_gram_schmidt(A)
    times_cgs[ind] = time_result.average

    time_result = %timeit -o -q Q,R = qr_modified_gram_schmidt(A)
    times_mgs[ind] = time_result.average

    time_result = %timeit -o -q Q,R = qr_givens(A)
    times_givens[ind] = time_result.average

    if MamNakodenehoHouseholdera:
        time_result = %timeit -o -q Q,R = qr_householder(A)
        times_hh[ind] = time_result.average
    else:
        times_hh[ind] = np.nan


plt.loglog(n_values, times_cgs, 'o:', color='tab:red', markeredgecolor='k', label='CGS')
plt.loglog(n_values, times_mgs, 'o:', color='tab:blue', markeredgecolor='k', label='MGS')
plt.loglog(n_values, times_givens, 'o:', color='tab:green', markeredgecolor='k', label='Givens')
plt.loglog(n_values, times_hh, 'o:', color='orange', markeredgecolor='k', label='Householder')
plt.legend(); plt.xlabel(r'$n$'); plt.ylabel(r'cas'); plt.show()

<div style="border-left: 4px solid #8250df; background-color: #fbefff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**❗ ZÚ (semestrální projekt)**

Na základě výsledků odhadněte závislost délky výpočtu $QR$ faktorizace matice $A\in \mathbb{R}^{n\times 10}$ na jejím počtu řádků $n$ za použití Landau notace ($\mathcal{O}(\dotsc)$). Detailně svou odpověď zdůvodněte/podložte daty z grafu.
- Okomentujte/Porovnejte s teoretickými výsledky z přednášky.
- [Nápověda & hinty](#head-w03-hinty-b2)

</blockquote>
</div>

## Část C: Řešení soustav lineárních algebraických rovnic pomocí $QR$-faktorizace

Jedna z hlavních aplikací pro $QR$-faktorizaci matice je řešení soustav lineárních algebraických rovnic, tj.

\begin{equation*}
A\mathbf{x} = \mathbf{b}.
\end{equation*}

Stejně jako u [cvičení o iteračních metodách](week-02-iterative-methods) máme v principu dvě možnosti, které staví na QR faktorizaci.



### QR faktorizace $A$
> Tato metoda je vhodnější pokud víme, že chceme vyřešit několik lineárních systémů s maticí $A$, tj.
\begin{equation*} A\mathbf{x}_1 = \mathbf{b}_1 , \dotsc , A\mathbf{x}_N = \mathbf{b}_N, \end{equation*}
ale nemáme k dispozici všechny pravé strany $A\mathbf{b}_1, \dotsc , \mathbf{b}_1$. Pak se nám vyplatí si předpočítat faktory $Q$ a $R$ a následně pokaždé, když dostaneme další pravou stranu $\mathbf{b}_i$ ji transformujeme na $\tilde{\mathbf{b}}_i := Q^T\mathbf{b}_i$ a poté řešíme systém
>
> \begin{equation*} R\mathbf{x}_i = \tilde{\mathbf{b}}_i.\end{equation*}


### QR faktorizace $[A|\mathbf{b}]$
> Tato metoda je vhodnější pokud víme, že chceme vyřešit pouze jeden lineární systém s maticí $A$ a tedy není důvod si ukládat matici $Q$.
>
>Místo postupu výše můžeme aplikovat transformace (projektory v Gram-Schmidtovy, rotace u Givense nebo reflexe u Householdera) nejen na sloupce matice $A$ tak jak jsme zvyklí, ale zároveň i na vektor pravé strany $\mathbf{b}$. Mělo by být zjevné, že toto odpovídá QR faktorizaci matice
>
> \begin{equation*} \tilde{Q} \tilde{R} =  [A|\mathbf{b}] \in\mathbb{R}^{n\times (n+1)},
\qquad \mathrm{kde} \qquad \tilde{Q}\in \mathbb{R}^{n\times n} \; \mathrm{a} \;  \tilde{R}\in \mathbb{R}^{n\times (n+1)},
\end{equation*}
>
>podobně jako Gaussova eliminace odpovídá LU faktorizaci $[A|\mathbf{b}]$, kde $U$ faktor je "o jeden sloupec širší". Stejně tak i v tomto případě pak získáme řešení $\mathbf{x}$ původního problému řešením soustavy
>
>\begin{equation*} \tilde{R}_{1:n,1:n}\mathbf{x} = \tilde{R}_{:,n+1}.\end{equation*}

### Úloha C1

Uvažujme problém $A\mathbf{x}= \mathbf{b}$, kde $A\in \mathbb{R}^{n\times n}$ je regulární a zachovejme značení jako výše.

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

Ukažte, že oba postupy výše jsou matematicky ekvivalentní. Předpokládejme, že matice $R$ má kladné diagonální prvky. Pomocí maticových operací ukažte, že platí
  - $Q = \tilde{Q}$,
  - $R = \tilde{R}_{:,1:n}$,
  - $Q^T\mathbf{b} = \tilde{R}_{:,n+1}$,
  a  tudíž soustavy výše mají stejné řešení $\mathbf{x}\in \mathbb{R}^n$.

</blockquote>
</div>

### Úloha C2

Vezmeme si matici $A\in \mathbb{R}^{m\times n}$ a její $QR$-faktorizaci $A=QR$. V úloze C1 jsme viděli, že pokud jde o řešení soustav algebraických rovnic, pak "$R$ podědí všechny důležité informace po $A$" - stačí nám řešit pozměněnou soustavu pouze s $R$. Co dalšího ta matice $R$ po $A$ podědí?

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Tužka a papír**

Ukažte, že platí
  - $\|A\|_2 = \|R\|_2$ a také $\|A\|_F = \|R\|_F$,
  - $\kappa(A) = \kappa(R)$.

</blockquote>
</div>

### Úloha C3

Uvažujme problém $A\mathbf{x}= \mathbf{b}$, kde $A\in \mathbb{R}^{n\times n}$ a zachovejme značení jako výše. Vyzkoušíme, jestli "matematicky ekvivalentní" funguje i při reálném výpočtu.

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Kódění**

Nakóděte funkce `rel_error(x_true, x_approx)` a `rel_residual(A, b, x_approx)`, které počítají velikost relativní chyby $e_{rel}$ a relativního rezidua $r_{rel}$ daného numerické aproximace, viz anotace.

</blockquote>
</div>

In [ ]:
def rel_error(x_true, x_approx):
    """ Spocitame relativni primou chybu pro danou aproximaci "x_approx" a presne reseni "x_true". """
    # Vas kod
    return

def rel_residual(A, b, x_approx):
    """ Spocitame relativni reziduum (aka zpetnou chybu) pro danou aproximaci "x_approx" a problem zadany matici "A" a pravou stranou "b". """
    # Vas kod
    return

####################################################
def test_rel_ErrorResidual():
    x = np.arange(0,10,1)
    assert (rel_error(x, x) < 1e-10 ), "rel_error pro x_approx = x_true by melo vratit 0"
    y = x + 1
    assert ( rel_error(x, y) > 0 ), "rel_error by melo vzdy vracet kladne hodnoty pro x_approx != x_true"
    assert ( abs(rel_error(x, y) - np.sqrt(10) / np.linalg.norm(x)) < 1e-8 ), "rel_error je vypocitan spatne pro x_true = [1,2,3] a x_approx = [2,3,4]"

    n_values = np.array([5, 10, 20])
    for ind_n in range(len(n_values)):
        n = n_values[ind_n]
        A = np.random.normal(size=(n, n))
        x_true = np.random.normal(size=n)
        b = A @ x_true
        x_approx = x_true
        assert ( rel_residual(A, b, x_approx) < 1e-10 ), "rel_residual pro x_approx = x_true by mel vratit 0"
        x_approx = np.random.normal(size=n)
        assert ( rel_residual(A, b, x_approx) > 0 ), "rel_residual by mel vzdy vracet kladne hodnoty pro A*x_approx != b"
        A = np.eye(n)
        x_true = np.ones(n)
        b = A @ x_true
        x_approx = x_true + 1
        assert ( abs(rel_residual(A, b, x_approx) - 1) < 1e-8 ), "rel_residual pro A jednotkovou matici, b=[1,1,1,1,1], x_approx=[2,2,2,2,2] vraci spatny vysledek"
    return "'rel_error()' & 'rel_residual()' jsou OK"
test_rel_ErrorResidual()

<div style="border-left: 4px solid #0969da; background-color: #ddf4ff; padding: 0.75em 1em; border-radius: 4px; margin: 1em 0;">
<blockquote style="margin: 0; padding: 0; border: none;">

**📝 Úkol**

Pro obě QR faktorizace ($A$ i $[A|\mathbf{b}]$) a všechny 4 algoritmy doplňte výpočet `x_cgs`/`x_mgs`/`x_givens`/`x_hh` a odpovídající chyby `ef_*`/`eb_*` pomocí `rel_error()`/`rel_residual()`.

Stejně jako v B1 budeme generovat matice $A$ s postupně rostoucím číslem podmíněnosti $\kappa(A)$. Vykreslete jak se bude vyvíjet $e_{rel}$ a $r_{rel}$ podle zvoleného typu výpočtu $QR$-faktorizace.
  - vhodná volba testovacích hodnot $\kappa(A)$ je například `cond_numbers = np.logspace(0,20,30); n = 50`.
  - otestujte obě verze aplikace QR faktorizace, tj. jako QR faktorizaci $A$ tak $[A|\mathbf{b}]$,
  - pro sestavování matic a vektorů "po blocích" se hodí [`np.block()`](https://numpy.org/doc/stable/reference/generated/numpy.block.html), ale lze použít také [`np.concatenate()`](https://numpy.org/doc/stable/reference/generated/numpy.concatenate.html).

Výsledky pečlivě okomentujte:
    - Jsou všechny zvolené přístupy ekvivalentní i pro reálné výpočty?
    - Na čem odpověď závisí?
    - Jak ne/odpovídá odpověď teorii z přednášky?

</blockquote>
</div>

In [ ]:
def random_matrix(n, cond):
    """
    Vrati nesingularni ctvercovou matici A rozmeru "n"-krat-"n" s cislem podminenosti kappa(A) = "cond".
    """
    A = rng.standard_normal(size=(n, n))
    U, _, Vt = np.linalg.svd(A)
    S = np.diag(np.logspace(0, -np.log10(cond), n))
    return U @ S @ Vt

cond_numbers = np.logspace(0, 12, 20)
n = 50

### inicializace pro qr(A)
err_cgs_a = np.zeros(len(cond_numbers))
res_cgs_a = np.zeros(len(cond_numbers))
err_mgs_a = np.zeros(len(cond_numbers))
res_mgs_a = np.zeros(len(cond_numbers))
err_givens_a = np.zeros(len(cond_numbers))
res_givens_a = np.zeros(len(cond_numbers))
err_hh_a = np.zeros(len(cond_numbers))
res_hh_a = np.zeros(len(cond_numbers))

### inicializace pro qr([A|b])
err_cgs_ab = np.zeros(len(cond_numbers))
res_cgs_ab = np.zeros(len(cond_numbers))
err_mgs_ab = np.zeros(len(cond_numbers))
res_mgs_ab = np.zeros(len(cond_numbers))
err_givens_ab = np.zeros(len(cond_numbers))
res_givens_ab = np.zeros(len(cond_numbers))
err_hh_ab = np.zeros(len(cond_numbers))
res_hh_ab = np.zeros(len(cond_numbers))


for ind in range(len(cond_numbers)):
    A = random_matrix(n, cond_numbers[ind])
    x_true = rng.standard_normal(size=(n))
    b = A @ x_true
    A_aug = np.concatenate((A, b.reshape((len(b), 1))), axis=1)

    ### vypocet zalozeny na "qr(A)"
    ...

    ### vypocet zalozeny na "qr([A|b])"
    ...


fig, [[ax1, ax2], [ax3, ax4]] = plt.subplots(nrows=2, ncols=2, figsize=(16, 10), sharex='all')

ax1.loglog(cond_numbers, err_cgs_a, 'o:', color='tab:red', markeredgecolor='k', label=r'$e_{rel}$ CGS')
ax1.loglog(cond_numbers, err_mgs_a, 'o:', color='tab:blue', markeredgecolor='k', label=r'$e_{rel}$ MGS')
ax1.loglog(cond_numbers, err_givens_a, 'o:', color='tab:green', markeredgecolor='k', label=r'$e_{rel}$ Givens')
ax1.loglog(cond_numbers, err_hh_a, 'o:', color='orange', markeredgecolor='k', label=r'$e_{rel}$ Householder')
ax1.legend(); ax1.set_title(r'QR faktorizace $A$')

ax2.loglog(cond_numbers, res_cgs_a, 'o:', color='tab:red', markeredgecolor='k', label=r'$r_{rel}$ CGS')
ax2.loglog(cond_numbers, res_mgs_a, 'o:', color='tab:blue', markeredgecolor='k', label=r'$r_{rel}$ MGS')
ax2.loglog(cond_numbers, res_givens_a, 'o:', color='tab:green', markeredgecolor='k', label=r'$r_{rel}$ Givens')
ax2.loglog(cond_numbers, res_hh_a, 'o:', color='orange', markeredgecolor='k', label=r'$r_{rel}$ Householder')
ax2.legend(); ax2.set_title(r'QR faktorizace $A$')

ax3.loglog(cond_numbers, err_cgs_ab, 'o:', color='tab:red', markeredgecolor='k', label=r'$e_{rel}$ CGS')
ax3.loglog(cond_numbers, err_mgs_ab, 'o:', color='tab:blue', markeredgecolor='k', label=r'$e_{rel}$ MGS')
ax3.loglog(cond_numbers, err_givens_ab, 'o:', color='tab:green', markeredgecolor='k', label=r'$e_{rel}$ Givens')
ax3.loglog(cond_numbers, err_hh_ab, 'o:', color='orange', markeredgecolor='k', label=r'$e_{rel}$ Householder')
ax3.legend(); ax3.set_xlabel(r'$\kappa(A)$'); ax3.set_title(r'QR faktorizace $[A|b]$')

ax4.loglog(cond_numbers, res_cgs_ab, 'o:', color='tab:red', markeredgecolor='k', label=r'$r_{rel}$ CGS')
ax4.loglog(cond_numbers, res_mgs_ab, 'o:', color='tab:blue', markeredgecolor='k', label=r'$r_{rel}$ MGS')
ax4.loglog(cond_numbers, res_givens_ab, 'o:', color='tab:green', markeredgecolor='k', label=r'$r_{rel}$ Givens')
ax4.loglog(cond_numbers, res_hh_ab, 'o:', color='orange', markeredgecolor='k', label=r'$r_{rel}$ Householder')
ax4.legend(); ax4.set_xlabel(r'$\kappa(A)$'); ax4.set_title(r'QR faktorizace $[A|b]$')

plt.show()

## Hinty

<a name="head-w03-hinty-b2"></a>
### Hinty B2

Předpokládejme, že máme funkci $y = g(x)>0$, kterou jsme vykreslili v logaritmickém měřítku na obou osách. Pokud vidíme, že takový graf je rostoucí přímka s kladnými hodnotami, jaký vztah platí mezi $\log_{10} (g(x))$ a $\log_{10}(x)$? Jak pak vypadá $g(x)$? Porovnejte s [Úlohou A2 cvičení o iteračních metodách](week-02-iterative-methods).